# NeuralAtmosphereOperator — từ FNO đến SFNO và thiết kế model

> **Phạm vi của notebook.** Tài liệu mô tả bài toán dưới góc nhìn neural operator, nền tảng FNO, lý do chuyển sang SFNO, kiến trúc được triển khai và quá trình đi từ **loss ban đầu** đến **training objective hiện tại**. Phần cuối xác định luôn nguyên tắc đánh giá cần thiết để validation/test có cùng ý nghĩa với train loss. Các chi tiết vận hành như DataLoader, optimizer, checkpoint và deployment vẫn nằm trong runbook riêng.

Model được mô tả là baseline hiện tại của repository: **SFNO-SC2-L6-E128**, nhận một trạng thái khí quyển toàn cầu 26 kênh trên lưới $361\times720$ và dự báo trạng thái kế tiếp. Đây là một biến thể nhỏ gọn thuộc cùng họ SFNO, không phải bản sao nguyên xi của cấu hình lớn trong Makani.

Notebook cố ý giữ lại thiết kế loss trước khi audit. Mục đích là ghi lại đầy đủ chuỗi suy luận: vì sao công thức cũ từng có vẻ hợp lý, phép đổi đơn vị nào làm lộ ra điểm chưa nhất quán, bằng chứng khoa học nào dẫn đến công thức mới, và các metric nào phải theo dõi để không chọn checkpoint chỉ từ một scalar thiếu ngữ cảnh.

> **Quy ước thuật ngữ.** Notebook giữ nguyên những thuật ngữ tiếng Anh đã phổ biến trong machine learning và scientific computing—chẳng hạn *embedding*, *spectral convolution*, *mode truncation*, *residual connection*, *rollout* và *loss*—khi bản dịch tiếng Việt làm sai sắc thái kỹ thuật hoặc khiến câu văn thiếu tự nhiên. Thuật ngữ được giải thích tại lần xuất hiện đầu tiên.

Nguồn chính: [Li et al., Fourier Neural Operator, ICLR 2021](https://arxiv.org/abs/2010.08895), [Bonev et al., Spherical Fourier Neural Operators, ICML 2023](https://proceedings.mlr.press/v202/bonev23a.html), [Lam et al., GraphCast, Science 2023](https://doi.org/10.1126/science.adi2336), mã tham chiếu [NVIDIA Makani](https://github.com/NVIDIA/makani), và phiên bản triển khai đã khóa [NVIDIA torch-harmonics v0.7.4](https://github.com/NVIDIA/torch-harmonics/tree/v0.7.4).


## 1. Lộ trình lập luận

Tài liệu đi theo thứ tự sau:

1. Biểu diễn dự báo khí quyển như một ánh xạ giữa các **trường vật lý**.
2. Từ neural operator tổng quát đến **Fourier Neural Operator (FNO)** trên miền phẳng, tuần hoàn.
3. Chỉ ra vì sao FFT hai chiều không phù hợp hoàn toàn với hình học Trái Đất.
4. Thay planar Fourier basis bằng **spherical harmonics** để thu được SFNO.
5. Bóc tách kiến trúc SFNO cụ thể của dự án, từ tensor đầu vào đến đầu ra.
6. Giải thích từng lựa chọn cấu hình, số mode và số tham số.
7. Khôi phục loss ban đầu và phân tích nó trong cả normalized space lẫn physical space.
8. Từ điểm bất hợp lý của loss cũ, xây dựng standardized-tendency objective hiện tại.
9. Mở rộng objective theo curriculum $K=1\rightarrow2\rightarrow4\rightarrow8$ và đồng bộ train/validation/test.
10. Xác định cách đọc loss cùng RMSE, ACC và baseline để chọn checkpoint.

Thứ tự này quan trọng: SFNO không phải một mạng hoàn toàn tách biệt với FNO; nó thay planar Fourier transform trong FNO bằng một spectral transform phù hợp với miền $\mathbb S^2$. Tương tự, loss mới không xuất hiện như một lựa chọn tùy ý mà là kết quả của việc kiểm tra loss cũ theo đúng đơn vị mà optimizer thực sự nhìn thấy.


## 2. Bài toán dự báo dưới góc nhìn toán tử

Gọi trạng thái khí quyển tại thời điểm $t$ là trường nhiều kênh

$$
\mathbf{x}_t : \mathbb S^2 \rightarrow \mathbb R^{C}, \qquad C=26.
$$

Tại mỗi vị trí trên bề mặt cầu, $\mathbf{x}_t$ chứa 26 đại lượng bề mặt hoặc theo mực áp suất. Bài toán one-step forecasting là xấp xỉ toán tử tiến hóa

$$
\mathcal G_{\Delta t}: \mathbf{x}_t \mapsto \mathbf{x}_{t+\Delta t}.
$$

Một neural network thông thường xử lý tensor ảnh có thể được xem như ánh xạ giữa hai không gian hữu hạn chiều. Neural operator nhấn mạnh rằng đối tượng cần học về bản chất là ánh xạ giữa các **function spaces**:

$$
\mathcal G_\theta : \mathcal A \subseteq \{\mathbb S^2\to\mathbb R^{C_{in}}\}
\longrightarrow
\mathcal U \subseteq \{\mathbb S^2\to\mathbb R^{C_{out}}\}.
$$

Lưới $361\times720$ chỉ là một phép rời rạc hóa của các trường đó. Góc nhìn toán tử phù hợp với động lực học khí quyển vì cùng một quy luật tiến hóa phải tác động lên toàn bộ các trạng thái, thay vì ghi nhớ từng bản đồ riêng lẻ. Tuy vậy, một model học từ dữ liệu vẫn chỉ xấp xỉ toán tử trên phân bố và độ phân giải đã quan sát; không nên diễn giải đây là bộ giải PDE chính xác hoặc đảm bảo tổng quát hóa ở mọi lưới.

## 3. Neural operator tổng quát

Một lớp neural operator có thể được viết dưới dạng

$$
v_{l+1}(x)=\sigma\!\left(W_l v_l(x)+\left(\mathcal K_l v_l\right)(x)\right),
$$

với toán tử tích phân

$$
(\mathcal K_l v)(x)=\int_D \kappa_l(x,y) v(y)\,dy.
$$

Trong đó:

- $v_l(x)$ là hidden representation tại vị trí $x$;
- $W_l$ là biến đổi cục bộ theo kênh;
- $\kappa_l(x,y)$ là kernel học được, cho phép vị trí $x$ nhận thông tin từ vị trí $y$;
- $\sigma$ là nonlinear activation, giúp nhiều lớp liên tiếp mô tả được động lực học phức tạp.

Nếu tính tích phân trực tiếp trên mọi cặp $(x,y)$, chi phí tăng gần bậc hai theo số điểm lưới. Ý tưởng trung tâm của FNO là tham số hóa kernel và thực hiện global mixing trong miền Fourier, nơi convolution trở thành phép nhân.

# Phần I — Fourier Neural Operator (FNO)

## 4. Từ convolution đến spectral convolution

Trên miền phẳng tuần hoàn $\mathbb T^2=S^1\times S^1$, biến đổi Fourier của trường $v$ là

$$
\widehat v(k_1,k_2)=\int_{\mathbb T^2}v(x_1,x_2)
e^{-2\pi i(k_1x_1+k_2x_2)}\,dx_1dx_2.
$$

Convolution theorem cho phép viết

$$
\mathcal F[\kappa * v](k)=\widehat\kappa(k)\,\widehat v(k).
$$

FNO thay $\widehat\kappa(k)$ bằng tensor phức học được $R_\theta(k)$ và định nghĩa spectral convolution

$$
\mathcal K_\theta(v)=\mathcal F^{-1}\!\left(R_\theta\,\mathcal F(v)\right).
$$

Với nhiều kênh, tại mỗi mode $k$ ta thực hiện channel mixing

$$
\widehat z_o(k)=\sum_{c=1}^{C_{in}}R_{o,c}(k)\widehat v_c(k).
$$

Do đó một mode đầu ra có thể kết hợp thông tin từ mọi kênh đầu vào. Vì mỗi hệ số Fourier được tính từ toàn miền, một lớp spectral đã có receptive field toàn cục. Đây là ưu điểm quan trọng đối với các hệ có tương tác xa như hoàn lưu khí quyển.

## 5. Cấu trúc chuẩn của một FNO

Một FNO điển hình gồm ba giai đoạn:

### 5.1 Lifting

Ánh xạ số kênh vật lý $C_{in}$ sang embedding dimension $E$:

$$v_0(x)=P\mathbf{x}(x), \qquad P\in\mathbb R^{E\times C_{in}}.$$

Trong CNN, đây tương đương convolution $1\times1$: chỉ thực hiện channel mixing tại cùng vị trí, chưa thực hiện spatial mixing.

### 5.2 Các Fourier block

Dạng khái niệm thường gặp là

$$v_{l+1}=\sigma\left(W_l v_l+\mathcal K_{\theta_l}v_l\right).$$

Nhánh spectral đảm nhiệm tương tác toàn cục; nhánh pointwise/skip giữ thông tin cục bộ và tạo đường truyền gradient. Các implementation hiện đại có thể đặt normalization, MLP và residual theo block topology khác nhau, nhưng nguyên lý spectral vẫn không đổi.

### 5.3 Projection

Ánh xạ embedding trở lại $C_{out}$ kênh dự báo:

$$\widehat{\mathbf y}(x)=Qv_L(x), \qquad Q\in\mathbb R^{C_{out}\times E}.$$

## 6. Mode truncation: vì sao FNO không học toàn bộ phổ

FNO thường chỉ giữ tập mode tần số thấp $\Lambda$:

$$
\widehat z(k)=
\begin{cases}
R_\theta(k)\widehat v(k), & k\in\Lambda,\\
0, & k\notin\Lambda.
\end{cases}
$$

Spectral truncation có ba tác dụng:

- giảm số tham số và chi phí;
- tập trung capacity vào cấu trúc quy mô lớn, thường chứa phần lớn năng lượng của trường trơn;
- tạo một bottleneck phổ có tác dụng regularization.

Mode truncation cũng đặt ra giới hạn biểu diễn: chi tiết nhỏ hơn bước sóng tương ứng với mode lớn nhất không thể được truyền tuyến tính qua nhánh spectral. Pointwise MLP có thể biến đổi và tái phân phối phổ thông qua nonlinear activation, nhưng không làm biến mất giới hạn capacity do spectral bandwidth gây ra. Vì thế số mode là một tham số kiến trúc thực sự, không chỉ là mẹo tăng tốc.

## 7. Vì sao planar FNO chưa đủ cho khí quyển toàn cầu?

Một FFT 2D trên ma trận latitude–longitude ngầm xem hai trục là tuần hoàn độc lập, tức miền có topology của một torus $\mathbb T^2$. Điều đó đúng với longitude nhưng sai với latitude:

- kinh tuyến $0^\circ$ và $360^\circ$ nối nhau;
- Bắc Cực và Nam Cực là hai **điểm**, không phải hai đường biên tuần hoàn;
- khoảng cách vật lý theo longitude co lại theo $\cos\varphi$ khi tiến về cực;
- các ô latitude–longitude có diện tích không đồng đều;
- translation trên ảnh chữ nhật không tương đương rotation trên mặt cầu.

Nếu trực tiếp dùng FFT 2D, model phải học trên một hình học sai: các hàng ở cực bị đối xử như những vòng tròn có chu vi ngang hàng xích đạo, và điều kiện biên latitude trở thành giả tạo. Bonev et al. chỉ ra rằng sự không tương thích này có thể tạo spectral artifacts, làm tiêu tán năng lượng và làm rollout dài hạn kém ổn định.

Vấn đề không phải FFT là một phép biến đổi xấu; vấn đề là **planar Fourier basis không phải eigenbasis tự nhiên của mặt cầu**. Muốn giữ ý tưởng FNO nhưng sửa hình học, ta cần thay nó bằng spherical harmonic basis.

# Phần II — Từ FNO đến Spherical FNO

## 8. Spherical harmonics: Fourier basis của mặt cầu

Dùng colatitude $\theta\in[0,\pi]$ và longitude $\lambda\in[0,2\pi)$, spherical harmonics có dạng

$$
Y_\ell^m(\theta,\lambda)=N_{\ell m}P_\ell^m(\cos\theta)e^{im\lambda},
$$

trong đó $P_\ell^m$ là associated Legendre polynomial, $\ell\ge0$ là degree và $|m|\le\ell$ là order. Chúng tạo cơ sở trực chuẩn của $L^2(\mathbb S^2)$ dưới phần tử diện tích

$$d\Omega=\sin\theta\,d\theta\,d\lambda.$$

Spherical Harmonic Transform (SHT) và biến đổi ngược là

$$
\widehat f_{\ell m}=\int_{0}^{2\pi}\int_{0}^{\pi}
f(\theta,\lambda)\overline{Y_\ell^m(\theta,\lambda)}
\sin\theta\,d\theta\,d\lambda,
$$

$$
f(\theta,\lambda)=\sum_{\ell=0}^{\infty}
\sum_{m=-\ell}^{\ell}\widehat f_{\ell m}Y_\ell^m(\theta,\lambda).
$$

Tương tự Fourier modes trên đường tròn, $\ell$ biểu diễn spatial scale: $\ell$ nhỏ tương ứng với cấu trúc quy mô hành tinh, còn $\ell$ lớn tương ứng với cấu trúc nhỏ hơn. Khác biệt quyết định là basis này phản ánh metric và topology của $\mathbb S^2$.

## 9. Spherical spectral convolution

SFNO giữ nguyên tinh thần của FNO:

$$
\text{spatial field}
\xrightarrow{\mathrm{SHT}}
\text{spherical coefficients}
\xrightarrow{\text{learned mixing}}
\text{new coefficients}
\xrightarrow{\mathrm{SHT}^{-1}}
\text{spatial field}.
$$

Với `operator_type = driscoll-healy`, implementation dùng một ma trận phức $A_\ell$ cho mỗi degree $\ell$ và chia sẻ nó qua mọi order $m$:

$$
\widehat z_{o,\ell m}
=\sum_{c=1}^{E} A_{o,c,\ell}\widehat v_{c,\ell m},
\qquad A_\ell\in\mathbb C^{E\times E}.
$$

Trọng số được chia sẻ theo $m$ trong nhánh $m\geq0$. Với kernel phức và inverse real FFT, điều này không bảo đảm isotropic/SO(3)-equivariant scalar convolution: nhánh âm dùng liên hợp kernel, còn $m=0$ loại phần ảo. Đây là spherical spectral parameterization có symmetry hạn chế. Việc chia sẻ làm giảm số spectral weights từ bậc $O(E^2L^2)$ của kernel phụ thuộc cả $(\ell,m)$ xuống $O(E^2L)$.

Trong code v0.7.4, tensor spectral weight có shape

$$[E_{out},E_{in},L]$$

và dtype `complex64`. Vì vậy một complex tensor element chứa hai real-valued degrees of freedom.

## 10. SHT được tính như thế nào?

`torch-harmonics` triển khai SHT khả vi bằng hai bước chính:

1. FFT theo longitude để chiếu lên $e^{im\lambda}$;
2. quadrature theo latitude để chiếu lên associated Legendre polynomials $P_\ell^m$.

Đây là thuật toán “semi-naive” được thảo luận trong paper SFNO. Toàn bộ phép biến đổi được xây bằng các PyTorch primitives nên gradient có thể truyền qua SHT, spectral multiplication và inverse SHT.

Cần phân biệt hai khái niệm:

- **SHT không phải một lớp học được**: các bảng Legendre/quadrature là cấu trúc toán học cố định;
- **spectral kernel mới là phần học được**: các ma trận phức $A_\ell$ trộn kênh tại từng degree.

Nhờ vậy model dành tham số cho động lực học cần học, còn spherical geometry được encode trực tiếp trong transform. Nền tảng sampling/convolution trên cầu có liên hệ với [Driscoll & Healy, 1994](https://doi.org/10.1006/aama.1994.1008).
Bản sửa numerical SHT tách thành phần hằng bằng anchored centering: $\mathcal S(x)=\mathcal S(x-c)+c\sqrt{4\pi}e_{00}$ với harmonic orthonormal. Phần dư của input hằng bằng zero chính xác, tránh leakage float32 lên degree khác rồi bị InstanceNorm khuếch đại. Không thêm learned parameters hoặc đổi epsilon; near-constant sensitivity và CUDA vẫn cần kiểm chứng riêng.


# Phần III — Kiến trúc NeuralAtmosphereOperator

## 11. Sơ đồ tổng thể

Với $B$ là batch size, model thực hiện ánh xạ:

$$
\underbrace{[B,26,361,720]}_{\mathbf{x}_t}
\xrightarrow{1\times1\;\text{lifting}}
[B,128,361,720]
\xrightarrow{6\;\text{SFNO blocks}}
[B,128,361,720]
\xrightarrow{1\times1\;\text{projection}}
\underbrace{[B,26,361,720]}_{\Delta\mathbf{x}_t}
\xrightarrow{+\mathbf{x}_t}
\underbrace{[B,26,361,720]}_{\widehat{\mathbf{x}}_{t+\Delta t}}.
$$

Ba tầng chức năng là:

- **Lifting/encoder:** ánh xạ 26 kênh vật lý thành 128 hidden features.
- **SFNO backbone:** thực hiện sáu lần global mixing trên spherical harmonic modes, xen kẽ pointwise MLP và residual connection.
- **Projection/decoder:** ánh xạ hidden features về 26 tendency channels. Wrapper bên ngoài cộng tendency vào trạng thái hiện tại.

Implementation dự án nằm tại [`models/model.py`](../src/neural_atmosphere_operator/models/model.py); cấu hình khóa nằm tại [`configs/model_config.py`](../configs/model_config.py).

## 12. Danh sách và thứ tự 26 input/output channels

Channel dimension không phải 26 features tùy ý mà là một state vector có thứ tự cố định. Sáu surface channels là:

$$
(u_{10m},v_{10m},T_{2m},p_s,p_{msl},TCWV).
$$

Hai mươi pressure-level channels gồm:

- geopotential tại 1000, 850, 500, 250 và 50 hPa;
- zonal wind $u$ tại 1000, 850, 500 và 250 hPa;
- meridional wind $v$ tại 1000, 850, 500 và 250 hPa;
- temperature tại 850, 500, 250 và 100 hPa;
- specific humidity tại 1000 và 850 hPa;
- relative humidity tại 500 hPa.

Encoder đưa cả 26 kênh vào cùng embedding space 128 chiều, nên sau lifting không còn hidden feature nào tương ứng duy nhất với một biến vật lý. Decoder khôi phục đúng thứ tự 26 kênh; phép cộng residual chỉ đúng khi input state và output tendency dùng cùng channel ordering.

SFNO hiện tại xử lý mọi channel như một scalar field trong spectral mixing. Hai thành phần gió $u,v$ là vector components theo local basis, nhưng model không dùng vector spherical harmonics và không áp vector transformation law dưới rotation. Đây là một giới hạn quan trọng: kiến trúc nhận biết spherical geometry tốt hơn planar FFT, nhưng không phải một tensor/vector-field equivariant operator nghiêm ngặt.

## 13. Luồng tensor qua sáu SFNO blocks

`scale_factor = 2` tạo lưới nội bộ

$$
H_i=\frac{H-1}{2}+1=181,
\qquad W_i=\frac{W}{2}=360.
$$

Luồng chính xác là:

| Vị trí | Forward transform | Inverse transform | Output grid |
|---|---|---|---|
| Block 1 | equiangular $361\times720$ | Legendre–Gauss $181\times360$ | internal |
| Block 2–5 | Legendre–Gauss $181\times360$ | Legendre–Gauss $181\times360$ | internal |
| Block 6 | Legendre–Gauss $181\times360$ | equiangular $361\times720$ | full resolution |

Điểm cần nhấn mạnh: đây **không phải** lấy mỗi điểm thứ hai rồi bỏ dữ liệu trước khi model nhìn thấy nó. Block đầu đọc toàn bộ trường $361\times720$, biến đổi nó sang spherical harmonic domain, giữ spectral bandwidth đã cấu hình, rồi tái tạo band-limited representation trên quadrature grid nhỏ hơn. Vì transform đi trước việc đổi grid, đây là spectral restriction/resampling có cấu trúc, không phải spatial decimation tùy tiện.

Legendre–Gauss grid nội bộ phù hợp với quadrature SHT và tiết kiệm activation cho bốn block giữa. Lưới equiangular chỉ được giữ ở biên vào/ra để khớp tensor khí quyển.

## 14. Bên trong một SFNO block của phiên bản 0.7.4

Block topology thực tế không nên bị giản lược thành một FNO block chung chung. Với đầu vào $v_l$, mỗi block thực hiện:

$$
s_l=\operatorname{ISHT}_l
\left(A_l\odot\operatorname{SHT}_l(v_l)\right),
$$

$$
u_l=\operatorname{Norm}_0(s_l),
$$

$$
m_l=W_{2,l}\,\operatorname{GELU}(W_{1,l}u_l+b_{1,l}),
$$

$$
v_{l+1}=r_l+\operatorname{DropPath}\!\left(\operatorname{Norm}_1(m_l)\right).
$$

$r_l$ là residual branch của block input, được spectral-resample khi kích thước vào và ra khác nhau. Cấu hình hiện tại dùng:

- không có inner skip;
- outer skip là identity;
- hai affine InstanceNorm trong mỗi block;
- MLP $128\rightarrow256\rightarrow128$ với GELU;
- `drop_rate = 0` và `drop_path_rate = 0`, nên hai phép dropout hiện là identity.

Outer residual connection duy trì đường truyền thông tin và gradient, đồng thời cho phép block học correction so với representation trước đó. Pointwise MLP thực hiện channel mixing tại mỗi điểm sau global spectral mixing; nhờ nonlinear activation GELU, chuỗi block không suy biến thành một toán tử tuyến tính duy nhất.

## 15. Residual prediction ở cấp trạng thái

Gọi trạng thái vật lý, trạng thái đã chuẩn hóa và độ lớn tendency một bước của kênh $c$ lần lượt là

$$
z_{t,c}=\frac{x_{t,c}-\mu_c}{\sigma_{x,c}},
\qquad
\sigma_{\Delta,c}=\operatorname{Std}(x_{t+\Delta t,c}-x_{t,c}),
\qquad
r_c=\frac{\sigma_{\Delta,c}}{\sigma_{x,c}}.
$$

Backbone hiện tại không trực tiếp trả về trạng thái kế tiếp. Nó học **standardized tendency**

$$
\widehat d_{t,c}=F_\theta(\mathbf z_t)_c,
$$

rồi wrapper đổi tendency về đơn vị của normalized state trước khi cộng residual:

$$
\widehat z_{t+\Delta t,c}
=z_{t,c}+r_c\widehat d_{t,c}.
$$

Trong physical space, công thức tương đương là

$$
\widehat x_{t+\Delta t,c}
=x_{t,c}+\sigma_{\Delta,c}\widehat d_{t,c}.
$$

Cách tham số hóa này tách hai vai trò: $\sigma_{x,c}$ chuẩn hóa input state, còn $\sigma_{\Delta,c}$ xác định đơn vị tự nhiên của output tendency. Nếu backbone sinh zero tendency, dự báo trở thành persistence forecast.

Dự án tắt `big_skip` bên trong `torch-harmonics` và chỉ cộng residual một lần trong wrapper để tránh **double residual**. Khi đầu vào chứa nhiều trạng thái lịch sử ghép kênh, wrapper lấy 26 kênh cuối làm trạng thái hiện tại; `in_channels` vì thế phải là bội dương của `out_channels`.

Ánh xạ được lặp tự hồi quy mà không `detach` trạng thái trung gian:

$$
\widehat{\mathbf z}_{t+(k+1)\Delta t}
=\mathcal G_\theta\!\left(\widehat{\mathbf z}_{t+k\Delta t}\right).
$$

Vì đồ thị tính toán được giữ nguyên, loss ở lead sau có thể truyền gradient ngược qua tất cả các bước trước đó.


## 16. Cấu hình model đã chốt

| Tham số | Giá trị | Vai trò |
|---|---:|---|
| `img_size` | `(361, 720)` | Lưới toàn cầu 0.5°, gồm cả hai cực |
| `in_channels` | `26` | Số kênh trạng thái đầu vào khi không ghép history |
| `out_channels` | `26` | Số standardized-tendency channels đầu ra |
| `tendency_scale` | $r_c=\sigma_{\Delta,c}/\sigma_{x,c}$ | Đổi output tendency về đơn vị normalized state trước residual update |
| `scale_factor` | `2` | Lưới nội bộ $181\times360$; ký hiệu SC2 |
| `embed_dim` | `128` | Embedding dimension; ký hiệu E128 |
| `num_layers` | `6` | Sáu SFNO blocks; ký hiệu L6 |
| `operator_type` | `driscoll-healy` | Kernel phụ thuộc $\ell$, chia sẻ qua $m$ |
| `grid` | `equiangular` | Grid geometry của tensor vào/ra |
| `grid_internal` | `legendre-gauss` | Quadrature grid nội bộ |
| `activation_function` | `gelu` | Phi tuyến trong MLP |
| `normalization_layer` | `instance_norm` | Ổn định activation theo từng sample và feature channel |
| `use_mlp` | `True` | Bổ sung pointwise nonlinear channel mixing |
| `mlp_ratio` | `2.0` | MLP hidden dimension $=256$ |
| `hard_thresholding_fraction` | `1.0` | Giữ toàn bộ 180 mode khả dụng sau SC2 |
| `use_complex_kernels` | `True` | Dùng complex-valued spectral weights |
| `pos_embed` | `none` | Không thêm embedding vị trí học được |
| `drop_rate`, `drop_path_rate` | `0.0`, `0.0` | Baseline không stochastic regularization trong backbone |
| `use_residual_connection` | `True` | Dự báo standardized tendency rồi cộng trạng thái hiện tại qua $r_c$ |

Tên rút gọn đầy đủ là **SFNO-SC2-L6-E128**. `tendency_scale` được tạo từ statistics của đúng training split và đúng temporal stride; nó là một buffer của checkpoint contract, không phải tham số học được.


## 17. Số spherical modes thực sự được giữ

Với lưới nội bộ $181\times360$, implementation tính

$$
L_{lat}=181,
\qquad
L_{lon}=\left(\left\lfloor\frac{360}{2}\right\rfloor+1\right)-1=180.
$$

Số retained modes là

$$
L=\left\lfloor
\min(L_{lat},L_{lon})\,f_{HT}
\right\rfloor
=\lfloor180\times1.0\rfloor=180.
$$

`hard_thresholding_fraction = 1.0` nghĩa là không cắt thêm mode nào **bên trong SC2 bandwidth**. Nó không có nghĩa là giữ toàn bộ phổ mà lưới ngoài $361\times720$ có thể biểu diễn. Spectral bottleneck chính đã được xác định bởi internal grid và `scale_factor=2`.

Đây là một trade-off có chủ ý: 180 degree modes vẫn cung cấp global spectral mixing với capacity đáng kể, đồng thời giảm mạnh kích thước activation và transform so với việc giữ full-resolution bandwidth trong cả sáu block.

## 18. Phân bổ và cách tính chính xác số tham số

### 18.1 Spectral kernels

Mỗi block có tensor phức $[128,128,180]$:

$$N_{spec}=6\times128\times128\times180=17{,}694{,}720$$

**complex tensor elements**. Vì mỗi `complex64` gồm phần thực và phần ảo, số real-valued degrees of freedom tương đương là

$$2N_{spec}=35{,}389{,}440.$$

### 18.2 MLP trong sáu block

Mỗi MLP $128\to256\to128$ có bias ở lớp đầu nhưng không có bias ở lớp hai:

$$N_{MLP/block}=128\times256+256+256\times128=65{,}792,$$

$$N_{MLP}=6\times65{,}792=394{,}752.$$

### 18.3 Lifting và projection

Hai convolution $1\times1$ đều không bias:

$$N_{proj}=26\times128+128\times26=6{,}656.$$

### 18.4 InstanceNorm

Mỗi block có hai InstanceNorm; mỗi norm có scale và shift cho 128 kênh:

$$N_{norm}=6\times2\times2\times128=3{,}072.$$

### 18.5 Tổng

PyTorch báo số tensor elements học được là

$$
N_{tensor}=17{,}694{,}720+394{,}752+6{,}656+3{,}072
=\boxed{18{,}099{,}200}.
$$

Nếu đếm từng real-valued scalar độc lập, complex weights phải được nhân đôi:

$$
N_{real\ DOF}=35{,}389{,}440+394{,}752+6{,}656+3{,}072
=\boxed{35{,}793{,}920}.
$$

Hai con số đều đúng nhưng trả lời hai câu hỏi khác nhau. `sum(p.numel())` cho **18.10M tensor elements**; nếu quy đổi complex parameters thành các scalar components thì model có **35.79M real-valued DOF**. Khoảng 98.8% real-valued DOF nằm trong spectral kernels, nên $E$, $L$ và số layer chi phối capacity mạnh hơn số kênh vào/ra.

## 19. Vì sao chọn E128–L6 thay vì cấu hình lớn hơn?

Spectral capacity xấp xỉ

$$N_{spec}\propto N_{layers}E^2L.$$

Do phụ thuộc bậc hai vào embedding dimension, tăng E128 lên E256 không chỉ “gấp đôi model” mà làm riêng số spectral parameters tăng gần bốn lần. E384–L8 còn lớn hơn nhiều và không phù hợp để chọn làm mặc định chỉ vì nó gần một reference configuration.

E128–L6 được chọn vì:

- 26 kênh ít hơn đáng kể so với các model thời tiết lớn hơn;
- độ phân giải 0.5° có bandwidth thấp hơn 0.25°;
- sáu block vẫn cung cấp nhiều vòng global mixing + nonlinear pointwise mixing;
- 180 modes không cắt thêm bởi hard thresholding;
- spectral kernels vẫn mang 35.39M real DOF, nên đây không phải model “siêu nhỏ”;
- cấu hình giữ khoảng trống tài nguyên cho activation toàn cầu, vốn có thể tốn bộ nhớ hơn bản thân weights.

Đây là lựa chọn baseline có kiểm soát, không phải tuyên bố rằng E128 chắc chắn tối ưu. Cấu hình E384–L8 trong repository chỉ là capacity ablation và không được mô tả như bản sao chính thức của Makani.

## 20. Vai trò của các lựa chọn còn lại

### GELU

GELU tạo phi tuyến trơn hơn ReLU và được dùng trong MLP của từng block. Nếu không có phi tuyến, chồng nhiều spectral linear operators và projection tuyến tính cuối cùng vẫn chỉ tạo một ánh xạ tuyến tính, không đủ mô tả động lực học khí quyển.

### InstanceNorm trong backbone

Mỗi InstanceNorm chuẩn hóa activation theo từng sample và từng feature channel trên hai chiều không gian, rồi áp affine scale/shift học được. Nó không lưu running statistics. Thành phần này ổn định activation scale bên trong sáu block; đây là **internal architectural normalization**, khác với tiền xử lý các biến vật lý và được nhắc ở đây chỉ vì nó là một lớp của model.

### Không positional embedding

SHT đã encode spherical geometry và baseline ưu tiên convolution dùng chung trọng số trên toàn cầu. Tắt positional embedding làm giảm tham số phụ thuộc vị trí tuyệt đối và tránh để model ghi nhớ location-specific bias qua một bản đồ học được. Đổi lại, model không nhận thêm tín hiệu vị trí tuyệt đối ngoài những gì có thể suy ra từ state fields và grid.

### Không dropout/drop-path

Baseline giữ backbone deterministic và đơn giản để kiểm chứng. Hai cơ chế vẫn có trong config schema nhưng giá trị zero biến chúng thành identity. Chúng chỉ nên được bật như ablation khi có bằng chứng overfitting.

### Native initialization

Dự án giữ initialization gốc của `torch-harmonics`. Không chạy generic reinitialization lên toàn model vì spectral complex weights, MLP và residual paths sử dụng gain riêng để kiểm soát variance.

# Phần IV — Từ loss ban đầu đến objective hiện tại


## 21. Yêu cầu nền tảng: MSE đều trên ma trận là sai hình học

Điểm đúng ngay từ thiết kế ban đầu là không xem mọi điểm latitude–longitude như đại diện cho cùng diện tích. Nếu tính

$$\frac{1}{BCHW}\sum_{b,c,i,j}(\widehat x_{bcij}-x_{bcij})^2,$$

các hàng gần cực bị cho trọng số quá lớn so với diện tích thật. Cả loss cũ và loss mới vì thế đều dùng exact grid-cell area weights.

Gọi biên latitude của hàng $i$ là $\varphi_{i-1/2}$ và $\varphi_{i+1/2}$. Bỏ hằng số longitude chung, diện tích hàng tỉ lệ với

$$
a_i=\left|\sin\varphi_{i+1/2}-\sin\varphi_{i-1/2}\right|.
$$

Sau khi chuẩn hóa $a_i$ về trung bình một, một MSE theo diện tích có dạng

$$
\mathcal E_c=\frac{1}{BHW}\sum_{b,i,j}
a_i(\widehat z_{bcij}-z_{bcij})^2.
$$

Dùng latitude-cell boundaries thay vì trực tiếp $\cos\varphi_i$ bảo đảm hai hàng chứa cực vẫn có diện tích half-cell dương. Đây cũng là cách mã loss GraphCast chính thức xử lý grid chứa đúng $\pm90^\circ$.


## 22. Thiết kế ban đầu: channel-weighted spherical MSE

Phiên bản đầu tiên nhận thấy 26 kênh có đơn vị và độ biến thiên rất khác nhau, nên xây dựng

$$
\mathcal L_{\mathrm{old}}=\sum_{c=1}^{26}\alpha_c\mathcal E_c.
$$

Quy tắc base weight khi đó đi theo recipe `auto` của Makani:

- pressure-level field tại $p$ hPa nhận base weight $0.001p$;
- 2 m temperature nhận $1.0$;
- surface wind và pressure-like field nhận $0.1$;
- kênh không phân loại có fallback $0.01$.

Sau khi chuẩn hóa $q_c$ sao cho $\sum_cq_c=1$, implementation cũ dùng

$$
\alpha_c=q_c\frac{\sigma_{x,c}}{\sigma_{\Delta,c}}=\frac{q_c}{r_c}.
$$

Model cũ đồng thời cộng trực tiếp output backbone vào normalized state:

$$
\widehat z_{t+\Delta t,c}=z_{t,c}+F_\theta(\mathbf z_t)_c.
$$

Thiết kế này có lý do ban đầu tương đối thuyết phục: spherical area weighting sửa geometry, pressure weights tạo ưu tiên theo tầng khí quyển, còn tỉ lệ $\sigma_x/\sigma_\Delta$ tăng ảnh hưởng của những biến thay đổi chậm. Vì vậy nó không phải một loss vô nghĩa hay không khả vi. Vấn đề chỉ lộ rõ khi viết toàn bộ objective về cùng một hệ đơn vị.


## 23. Điểm bất hợp lý lộ ra khi đổi về physical space

Do $z_c=(x_c-\mu_c)/\sigma_{x,c}$,

$$
(\widehat z_c-z_c)^2
=\frac{(\widehat x_c-x_c)^2}{\sigma_{x,c}^2}.
$$

Thay vào loss cũ cho ta

$$
\mathcal L_{\mathrm{old}}
=\sum_c q_c\frac{1}{\sigma_{x,c}\sigma_{\Delta,c}}
\operatorname{MSE}_{area}(\widehat x_c,x_c).
$$

Trong khi mục tiêu “đo forecast error bằng đơn vị tendency một bước” phải là

$$
\mathcal L_{\Delta}
=\sum_c q_c\frac{1}{\sigma_{\Delta,c}^2}
\operatorname{MSE}_{area}(\widehat x_c,x_c).
$$

Hai công thức khác nhau một hệ số

$$
\frac{\mathcal L_{\mathrm{old},c}}{\mathcal L_{\Delta,c}}
=\frac{\sigma_{\Delta,c}}{\sigma_{x,c}}=r_c.
$$

Với biến có $r_c\ll1$, loss cũ vẫn thiếu một thừa số $1/r_c$ để thực sự trở thành standardized-tendency MSE. Hơn nữa, $q_c$ được chuẩn hóa **trước** khi nhân $1/r_c$ rồi không chuẩn hóa lại, nên absolute loss scale phụ thuộc mạnh vào statistics của bộ dữ liệu. Scalar loss giữa hai lần thay dữ liệu hoặc đổi temporal stride vì thế khó so sánh.

Điểm cần sửa nằm ở sự kết hợp giữa output parameterization và loss scaling. Chỉ tăng channel weight tuyến tính không đảm bảo backbone học output có unit variance, và không tạo đúng inverse time-difference variance trong physical space.


## 24. Đối chứng khoa học và quyết định thiết kế mới

Quyết định mới dựa trên ba lớp bằng chứng có vai trò khác nhau:

1. **GraphCast** dự báo sai phân trạng thái, chuẩn hóa output bằng per-variable/per-level standard deviation của $x_{t+1}-x_t$, rồi cộng sai phân đã đổi đơn vị vào trạng thái hiện tại. Objective của họ dùng inverse variance của time differences, area weights, pressure/variable weights và trung bình loss qua các lead. Xem [GraphCast Supplementary Materials, Sections 3.7, 4.2–4.5](https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/graphcast-ai-model-for-faster-and-more-accurate-global-weather-forecasting/Learning_skillful_medium-range_global_weather_forecasting.pdf), [normalization wrapper](https://github.com/google-deepmind/weathernext/blob/7077d40a36db6541e3ed72ccaed1c0d202fa6014/graphcast/normalization.py), [weighted loss](https://github.com/google-deepmind/weathernext/blob/7077d40a36db6541e3ed72ccaed1c0d202fa6014/graphcast/losses.py) và [autoregressive wrapper](https://github.com/google-deepmind/weathernext/blob/7077d40a36db6541e3ed72ccaed1c0d202fa6014/graphcast/autoregressive.py).
2. **Paper SFNO** dùng geometric relative $L_2$ cho single-step, sau đó cộng loss ở từng autoregressive step và backpropagate qua toàn bộ unrolled sequence. Điều này xác nhận area-aware objective và full BPTT là phù hợp với SFNO, dù scalar base loss của paper không trùng GraphCast. Xem [Bonev et al., Appendix B.2, Equations 30–31](https://proceedings.mlr.press/v202/bonev23a/bonev23a.pdf).
3. **NVIDIA Makani** cung cấp tiền lệ sát kiến trúc nhất: cấu hình SFNO chính thức dùng squared `l2`, `channel_weights: auto` và `temp_diff_normalization: true`. Makani cũng gán weight $0.1$ cho `sp` và `tcwv`, hai surface variables không có trong output set GraphCast gốc. Xem [SFNO config](https://github.com/NVIDIA/makani/blob/main/config/sfnonet.yaml) và [channel-weight implementation](https://github.com/NVIDIA/makani/blob/main/makani/utils/losses/base_loss.py).

Từ đó dự án chọn **GraphCast-style standardized-tendency objective cho một SFNO compact**. Đây là một thiết kế có căn cứ, không phải tuyên bố tái tạo nguyên xi GraphCast hoặc loss nguyên bản của SFNO paper.


## 25. Objective hiện tại

Backbone dự báo $\widehat d_{t,c}$ và wrapper áp dụng

$$
\widehat z_{t+\Delta t,c}=z_{t,c}+r_c\widehat d_{t,c},
\qquad
r_c=\frac{\sigma_{\Delta,c}}{\sigma_{x,c}}.
$$

Loss tại lead $k$ là

$$
\mathcal L_k=
\sum_c q_c\frac{1}{BHW}\sum_{b,i,j}a_i
\left(
\frac{\widehat z^{(k)}_{b,c,i,j}-z^{(k)}_{b,c,i,j}}{r_c}
\right)^2.
$$

Vì

$$
\frac{\widehat z^{(k)}_c-z^{(k)}_c}{r_c}
=\frac{\widehat x^{(k)}_c-x^{(k)}_c}{\sigma_{\Delta,c}},
$$

objective trong physical space chính xác là

$$
\mathcal L_k=
\sum_c q_c\frac{1}{BHW}\sum_{b,i,j}a_i
\left(
\frac{\widehat x^{(k)}_{b,c,i,j}-x^{(k)}_{b,c,i,j}}
{\sigma_{\Delta,c}}
\right)^2.
$$

Các $q_c$ hiện tại được tạo như sau:

- mỗi pressure variable nhận tổng base weight bằng một; các level bên trong variable được chia theo $p/\sum p$;
- `2m_temperature` nhận weight tương đối $1.0$;
- surface wind, pressure-like fields và `tcwv` nhận $0.1$;
- toàn bộ vector cuối được chuẩn hóa về $\sum_cq_c=1$.

Chuẩn hóa tổng weight chỉ nhân toàn bộ objective với một hằng số dương, giữ nguyên relative channel weights và hướng gradient trước các cơ chế phụ thuộc norm như clipping. Nó đồng thời làm scalar loss dễ diễn giải ổn định hơn giữa các stage dùng cùng statistics.

Tại $K=1$, target standardized tendency là

$$
d_{t,c}=\frac{x_{t+\Delta t,c}-x_{t,c}}{\sigma_{\Delta,c}},
$$

và công thức rút gọn đúng thành

$$
\mathcal L_1=\sum_cq_c\operatorname{MSE}_{area}(\widehat d_{t,c},d_{t,c}).
$$

Gradient đối với một output element là

$$
\frac{\partial\mathcal L_1}{\partial\widehat d_{b,c,i,j}}
=\frac{2q_ca_i}{BHW}(\widehat d_{b,c,i,j}-d_{b,c,i,j}),
$$

nên gradient descent luôn kéo standardized tendency về đúng target theo trọng số đã định.


## 26. Từ một bước đến curriculum autoregressive

Mục tiêu 48 giờ tương ứng tám bước 6 giờ, nhưng không buộc model chưa học dynamics một bước phải tối ưu ngay một graph tám bước dài. Training được chia thành các stage:

| Stage | Rollout $K$ | Vai trò | Learning rate |
|---|---:|---|---|
| 1 | 1 | Học tendency 6 giờ và cân bằng các kênh | Cao nhất, có warmup/decay |
| 2 | 2 | Fine-tune khi model bắt đầu nhìn thấy chính dự báo của nó | Thấp hơn stage 1 |
| 3 | 4 | Giảm error accumulation đến 24 giờ | Tiếp tục giảm |
| 4 | 8 | Fine-tune cuối cho horizon 48 giờ | Thấp nhất |

Tại stage $K$, objective là

$$
\mathcal L_K=
\frac{\sum_{k=1}^{K}\gamma^{k-1}\mathcal L_k}
{\sum_{k=1}^{K}\gamma^{k-1}},
\qquad 0<\gamma\le1.
$$

Baseline dùng $\gamma=1$, nghĩa là các lead trong cùng stage có trọng số thời gian bằng nhau. Với $k>1$, $\mathcal L_k$ nên được gọi chính xác là **forecast-state error tại lead $k$ đo theo đơn vị one-step tendency standard deviation**. Nó không còn đơn giản là sai số giữa hai tendency cùng bắt đầu từ ground-truth state.

Không có `detach` giữa các bước, nên gradient của $\mathcal L_2,\ldots,\mathcal L_K$ gồm cả tác động trực tiếp ở lead đó và tác động gián tiếp qua mọi trạng thái dự báo trước. Đây là full backpropagation through time, cùng nguyên tắc với autoregressive fine-tuning trong SFNO và GraphCast. Lịch $1\rightarrow2\rightarrow4\rightarrow8$ là adaptation phù hợp horizon 48 giờ và ngân sách của dự án; paper cung cấp nguyên tắc tăng dần rollout, không chứng minh riêng bốn mốc này là tối ưu phổ quát.


## 27. Đồng bộ train, validation và test

Một scalar chỉ so sánh được khi nó đại diện cho cùng bài toán. Vì vậy tại mỗi stage:

$$
K_{train}=K_{valid}=K_{test},
\qquad
\gamma_{train}=\gamma_{valid}=\gamma_{test},
$$

và cả ba split dùng cùng $q_c$, $a_i$, $r_c$ cùng công thức $\mathcal L_K$. Với stage $K=1$, validation/test cũng chỉ chấm đúng một bước. Khi sang $K=2,4,8$, checkpoint selection được reset theo objective mới; không so trực tiếp `best_loss` của hai stage khác $K$ như thể chúng là cùng một đại lượng.

Validation và test quét các forecast initializations bằng consecutive sliding windows trong từng split. Các windows chồng lấn giúp không bỏ sót mốc thời gian nhưng không độc lập thống kê; báo cáo nghiên cứu cuối cần block-bootstrap hoặc cách ước lượng uncertainty theo các block thời gian.

Checkpoint trong một stage được chọn bằng mean validation objective

$$
\operatorname{score}_{valid}=\mathcal L_K^{valid}.
$$

Test split chỉ dùng sau khi đã chốt checkpoint/hyperparameters. Ngoài objective đồng bộ, evaluation phải báo theo từng lead:

- physical latitude-weighted RMSE theo từng channel;
- ACC sau khi trừ climatology chỉ được tính từ training split;
- bias và variance ratio;
- skill so với persistence và training climatology trên cùng forecast starts;
- số lượng mẫu hữu hiệu và các trường hợp ACC không xác định.

[WeatherBench](https://doi.org/10.1029/2020MS002203) và [WeatherBench 2](https://arxiv.org/abs/2308.15560) dùng latitude-weighted RMSE/ACC và nhấn mạnh việc so sánh đồng nhất theo variable, lead time và ground truth. Vì vậy training objective dùng để tối ưu, còn RMSE/ACC/baselines trả lời câu hỏi forecast có thực sự hữu ích hay không.


## 28. Cách diễn giải loss và quyết định dừng

Loss hiện tại là dimensionless. Do $\sum_cq_c=1$ và area weights có mean bằng một:

- tại $K=1$, $\sqrt{\mathcal L_1}$ là weighted RMS error của tendency, đo theo số lần one-step tendency standard deviation;
- $\mathcal L_1\approx1$ nghĩa là weighted RMS error trung bình xấp xỉ một $\sigma_\Delta$;
- $\mathcal L_1<1$ là tín hiệu tốt hơn theo thước đo này, nhưng không tạo một ngưỡng chất lượng phổ quát;
- với $K>1$, scalar là trung bình các lead nên phải đọc thêm `lead_losses`, không để cải thiện lead gần che khuất suy giảm lead cuối.

Quy tắc dừng trong một stage dựa trên validation objective đã hội tụ, không dựa trên train loss thấp tuyệt đối. Cần can thiệp khi một trong các dấu hiệu sau kéo dài qua nhiều lần validation:

1. train loss tiếp tục giảm nhưng validation loss tăng: bắt đầu overfit;
2. mean validation loss giảm nhưng final-lead loss hoặc RMSE mục tiêu tăng: objective trung bình đang che giấu horizon quan trọng;
3. model không vượt persistence/climatology ở các biến và lead trọng tâm;
4. ACC giảm, bias tăng hoặc variance ratio co về dưới một dù MSE giảm: forecast có thể bị làm trơn;
5. gradient không finite hoặc clipping gần như mọi update: optimization không còn ở chế độ ổn định.

Absolute loss giữa thiết kế cũ và mới không được đặt cạnh nhau để kết luận model tốt hơn, vì hai scalar dùng đơn vị và weighting khác nhau. Chỉ so sánh checkpoint dưới cùng objective, cùng $K$, cùng statistics, cùng data windows; chất lượng khoa học cuối cùng được kết luận từ metric vật lý và baseline trên validation/test.


## 29. Các loss cũ và thành phần ablation còn lại

Module loss vẫn giữ các thành phần tổng quát phục vụ kiểm tra hồi quy hoặc nghiên cứu ablation:

$$
\mathcal L_{combined}
=\mathcal L_{spatial}+\lambda_{spec}\mathcal L_{spec}.
$$

Training baseline hiện tại dùng trực tiếp `StandardizedTendencyLoss`; các thành phần dưới đây không nằm trong primary objective.

### Planar spectral loss

`SpectralLoss` tính sai lệch trên các hệ số `rfft2` của ma trận lat–lon. Nó không phải spherical harmonic loss, không cô lập riêng high frequencies và, với L2, Parseval khiến nó gần tương đương spatial MSE khi được chuẩn hóa đúng. Bật planar spectral penalty sẽ làm geometry của objective thiếu nhất quán với lý do chọn SFNO, nên mặc định bằng zero.

### Channel-relative loss

Thành phần

$$
\mathcal L_{rel,c}=
\frac{\|\widehat x_c-x_c\|_{2,a}}
{\|x_c\|_{2,a}+\varepsilon}
$$

là base loss của paper SFNO gốc. Nó hữu ích cho ablation và diagnostic nhưng normalization theo target norm thay đổi theo từng sample/lead, trong khi primary objective cần channel scale cố định từ training-only tendency statistics để train/valid/test so sánh trực tiếp.

### Backward-only LossScaler

`LossScaler` giữ nguyên forward value nhưng scale gradient của mỗi kênh theo nghịch đảo spatial gradient norm. Nó thay đổi optimization dynamics dù scalar loss không đổi và có thể vô hiệu hóa chủ đích của explicit channel weights; vì vậy mặc định tắt.

### Legacy Makani-style weighting

`makani_auto_channel_weights` được giữ để tái hiện thí nghiệm cũ, nhưng không còn là lựa chọn primary của training CLI. Việc giữ code này cho phép audit checkpoint lịch sử mà không biến công thức cũ thành mô tả sai về baseline hiện tại.


## 30. Những gì model cố ý không chứa

Baseline hiện tại không có:

- orography channel;
- land–sea mask;
- solar zenith angle hoặc embedding chu kỳ ngày/năm;
- physics-informed constraint hay conservation penalty;
- post-processing corrector;
- learned positional embedding;
- spherical spectral loss bổ sung.

Đây không phải tuyên bố rằng các thành phần trên vô ích. Đây là quyết định giữ baseline có thể kiểm soát: mọi cải tiến bổ sung đều thay đổi giả thuyết khoa học, nguồn dữ liệu, loss hoặc cách kiểm chứng. Nếu model xuất hiện bias theo địa hình, đất–biển hay chu kỳ bức xạ, các thành phần này là hướng mở rộng hợp lý; nhưng chúng không nên được âm thầm gộp vào baseline rồi khiến nguyên nhân cải thiện/suy giảm không thể truy vết.

Do không có static forcing và astronomical forcing, SFNO hiện tại phải suy ra các dấu hiệu liên quan một cách gián tiếp từ 26 state fields. Vì `pos_embed=none` và Driscoll–Healy kernel chia sẻ trọng số theo rotation, model cũng không có position-specific parameter map học được để ghi nhớ địa hình cố định. Đây là giới hạn cần nêu rõ khi diễn giải kết quả.


## 31. Implementation invariants

Model config và wrapper áp đặt các điều kiện sau:

1. Tensor vào phải đúng shape $[B,C_{in},H,W]$.
2. $(H-1)$ phải chia hết cho `scale_factor` vì lưới latitude gồm cả hai cực.
3. $W$ phải chia hết cho `scale_factor`.
4. Grid ngoài phải là `equiangular`; grid nội bộ chỉ nhận `legendre-gauss` hoặc `equiangular`.
5. Residual prediction yêu cầu $C_{in}\ge C_{out}$ và $C_{in}$ là bội của $C_{out}$.
6. Backbone phải trả đúng $[B,C_{out},H,W]$ trước khi cộng tendency.
7. Chỉ initialization native được chấp nhận để không ghi đè transform-specific scaling.
8. Wrapper kiểm tra API `torch-harmonics` và bắt buộc có cách tắt residual nội bộ.

Các invariants này không chỉ là kiểm tra kiểu dữ liệu. Chúng bảo vệ các giả định hình học và ngăn những lỗi khó thấy như downsampling làm sai cách xử lý hai cực, cộng residual hai lần hoặc load một phiên bản SFNO có block topology khác. Repository vì thế khóa `torch-harmonics==0.7.4`; khi nâng phiên bản cần re-audit topology và numerical behavior.


## 32. Tóm tắt thiết kế

NeuralAtmosphereOperator bắt đầu từ FNO: học global convolution bằng cách biến đổi state fields sang spectral domain, trộn các modes bằng complex weights rồi biến đổi ngược. Planar FNO không mô hình hóa đúng topology và metric của latitude–longitude grid, nên Fourier transform được thay bằng SHT để tạo SFNO.

Baseline **SC2-L6-E128** dùng sáu spherical spectral blocks, 180 modes, Driscoll–Healy kernels, MLP ratio 2, GELU, InstanceNorm và standardized-tendency residual prediction. Nó có **18,099,200 PyTorch tensor elements**, tương đương **35,793,920 real-valued trainable scalars** do phần lớn trọng số là complex64.

Loss ban đầu đã giải quyết đúng spherical area nhưng chỉ nhân normalized-state MSE với $\sigma_x/\sigma_\Delta$. Khi đổi về physical units, công thức đó tạo $1/(\sigma_x\sigma_\Delta)$ thay vì inverse tendency variance $1/\sigma_\Delta^2$. Objective hiện tại sửa đồng thời output parameterization và loss: backbone dự báo standardized tendency, wrapper nhân $\sigma_\Delta/\sigma_x$ trước residual update, còn forecast error được chia cho cùng scale rồi bình phương.

Training đi theo curriculum $K=1\rightarrow2\rightarrow4\rightarrow8$ với learning rate giảm dần và full BPTT. Trong từng stage, train/validation/test dùng cùng $K$, cùng lead weighting và cùng objective. Checkpoint được chọn bằng validation objective, sau đó phải được kiểm tra bằng physical RMSE, ACC, bias, variance ratio và persistence/climatology baselines theo từng lead.

Thiết kế này bám theo time-difference normalization và multistep objective của GraphCast, theo area-aware full autoregressive backpropagation của SFNO, đồng thời dùng Makani để mở rộng surface weights cho channel subset của dự án. Nó là một giả thuyết khoa học rõ ràng và kiểm chứng được cho model compact, không phải tuyên bố rằng một công thức đã được chứng minh tối ưu cho mọi tập dữ liệu.


## 33. Tài liệu tham khảo

1. Z. Li, N. Kovachki, K. Azizzadenesheli, et al., **Fourier Neural Operator for Parametric Partial Differential Equations**, ICLR 2021. [arXiv:2010.08895](https://arxiv.org/abs/2010.08895).
2. B. Bonev, T. Kurth, C. Hundt, et al., **Spherical Fourier Neural Operators: Learning Stable Dynamics on the Sphere**, ICML 2023, PMLR 202:2806–2823. [PMLR](https://proceedings.mlr.press/v202/bonev23a.html) · [PDF](https://proceedings.mlr.press/v202/bonev23a/bonev23a.pdf).
3. R. Lam, A. Sanchez-Gonzalez, M. Willson, et al., **Learning skillful medium-range global weather forecasting**, Science 382(6677), 2023. [DOI](https://doi.org/10.1126/science.adi2336) · [Supplementary Materials](https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/graphcast-ai-model-for-faster-and-more-accurate-global-weather-forecasting/Learning_skillful_medium-range_global_weather_forecasting.pdf) · [official code](https://github.com/google-deepmind/weathernext/tree/7077d40a36db6541e3ed72ccaed1c0d202fa6014/graphcast).
4. S. Rasp, P. D. Dueben, S. Scher, et al., **WeatherBench: A Benchmark Data Set for Data-Driven Weather Forecasting**, JAMES 12(11), 2020. [DOI](https://doi.org/10.1029/2020MS002203).
5. S. Rasp, S. Hoyer, A. Merose, et al., **WeatherBench 2: A benchmark for the next generation of data-driven global weather models**, 2023. [arXiv:2308.15560](https://arxiv.org/abs/2308.15560) · [official framework](https://github.com/google-research/weatherbench2).
6. J. R. Driscoll and D. M. Healy, **Computing Fourier Transforms and Convolutions on the 2-Sphere**, Advances in Applied Mathematics 15(2), 1994. [DOI:10.1006/aama.1994.1008](https://doi.org/10.1006/aama.1994.1008).
7. NVIDIA, **Makani: AI-based weather and climate prediction**. [Repository](https://github.com/NVIDIA/makani) · [SFNO configuration](https://github.com/NVIDIA/makani/blob/main/config/sfnonet.yaml).
8. NVIDIA, **torch-harmonics: Differentiable signal processing on the sphere for PyTorch**. [Repository](https://github.com/NVIDIA/torch-harmonics) · [implementation v0.7.4](https://github.com/NVIDIA/torch-harmonics/blob/v0.7.4/torch_harmonics/examples/models/sfno.py).
9. Cấu hình và implementation của dự án: [`model_config.py`](../configs/model_config.py), [`model.py`](../src/neural_atmosphere_operator/models/model.py), [`loss.py`](../src/neural_atmosphere_operator/models/loss.py), [`forecast.py`](../src/neural_atmosphere_operator/pipeline/forecast.py), [`train.py`](../scripts/train.py), [`evaluate.py`](../scripts/evaluate.py).

---

**Quy ước khoa học:** công thức FNO/SFNO mô tả nguyên lý và block topology của phiên bản triển khai đã khóa. Công thức loss cũ được giữ như lịch sử thiết kế; công thức standardized-tendency và protocol đánh giá mới mới là baseline hiện hành. Những điểm lấy từ paper được phân biệt với adaptation riêng của dự án, đặc biệt là channel subset, lịch $K=1,2,4,8$ và horizon 48 giờ.
